# SofaScore Bundesliga rating × odds × lineup score

Creates optimizer-ready scores from Bundesliga average ratings, expected match points from odds, and configurable-weight LigaInsider, Kickbase, Kicker, and RotoWire starting chances. LigaInsider and Kicker alternatives use configurable geometric per-slot decay; repeated Kickbase alternatives are pooled as team-level evidence and normalized to an expected XI while remaining below displayed starters. The exported CSV includes the provider-specific chances, the injury penalty, and the final `score` column.

Confirmed SofaScore, LigaInsider, and RotoWire name matches are stored with Transfermarkt mappings in `data/reference/player_name_cross_references.csv`. Kickbase and Kicker display names resolve to a unique same-team KBStats last name, then first name; only shared-name or no-match candidates at 50% similarity or higher are shown for notebook confirmation and stored separately in `data/reference/kickbase_player_name_cross_references.csv`. Those mappings are scoped by provider, team, and KBStats player, so one provider's display label cannot override the other.

Set the notebook's QUES penalty, alternative-decay, and source-weight variables to tune scoring. A provider-level QUES reduction is applied before LigaInsider or Kickbase formation-slot alternatives are normalized.

Goalkeeper expected match points are `1.1 + raw expected match points * (0.6 / 3)`: raw 0 maps to 1.1, 1.5 maps to 1.4, and 3 maps to 1.7. Outfield players retain raw 0–3 values. The exported `expected_match_points` is the value used in the score. Rerun this notebook to generate updated scores.

Goalkeeper alternatives use decay **0.60** (relative weights **1, 0.6, 0.36, …**), with existing provider normalization and injury handling. Outfield alternatives retain their configured decay.


In [ ]:









import sys
from pathlib import Path

# Absolute starting-chance reduction for a `QUES` injury flag from a lineup source.
QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY = 0.15
# Each rank is 45% of the preceding alternative before its slot is normalized.
ALTERNATIVE_STARTING_CHANCE_DECAY = 0.45
GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY = 0.60
# Relative trust weights for active lineup sources; use 0 to disable one.
LINEUP_SOURCE_WEIGHTS = {'ligainsider': 4.0, 'kickbase': 3.0, 'kicker': 2.0, 'rotowire': 1.0}
# You will confirm whether each source is current for the selected matchday.

def locate_project_root() -> Path:
    starts = [Path.cwd().resolve()]
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.insert(0, Path(notebook_path).expanduser().resolve().parent)
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py.')

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sofascore_rating_odds_lineup_score import run_score_creation
result = run_score_creation(
    'bundesliga',
    questionable_injury_starting_chance_penalty=QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY,
    alternative_starting_chance_decay=ALTERNATIVE_STARTING_CHANCE_DECAY,
    goalkeeper_alternative_starting_chance_decay=GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY,
    lineup_source_weights=LINEUP_SOURCE_WEIGHTS,
)
